In [7]:
import numpy as np
import pandas as pd
from datetime import datetime
from pandas.tseries.offsets import MonthEnd
import sqlite3
import random

np.random.seed(42)
random.seed(42)

# ==============================
# 1) تعريف الثوابت والقيَم
# ==============================

ADMIN_AREA = "Upstream Admin Area - East"

DEPARTMENTS = [
    "UP Well Ops",
    "UP Gas Ops",
    "UP Maint Ops",
]

FACILITIES = [
    "Ghawar Area",
    "Abqaiq Area",
    "Khurais Area",
    "Wasit Gas Plant",
    "Uthmaniyah Gas Plant",
]

STREAMS = [f"UP-C{str(i).zfill(2)}" for i in range(1, 16)]

OPERATOR_NAMES = [
    "Ahmed Ali", "Faisal Al-Qahtani", "Saad Al-Mutairi",
    "Hassan Al-Dossari", "Mohammed Al-Otaibi", "Omar Al-Shehri"
]

NOTES_LIST = [
    "تشغيل مستقر.",
    "صيانة طفيفة مجدولة.",
    "تذبذب بسيط في الإنتاج بسبب الظروف التشغيلية.",
    "زيادة طفيفة عن الخطة.",
    "انخفاض بسيط عن الهدف، ضمن النطاق المقبول.",
    "Planned maintenance window.",
    "Operating within expected range.",
]

# متوسط حرارة تقريبية للمنطقة الشرقية (°C)
MONTH_TEMP = {
    1:  (15, 3),
    2:  (18, 3),
    3:  (22, 3),
    4:  (27, 3),
    5:  (33, 3),
    6:  (37, 3),
    7:  (40, 3),
    8:  (41, 3),
    9:  (37, 3),
    10: (30, 3),
    11: (24, 3),
    12: (18, 3),
}

YEARS = [2022, 2023, 2024]
MONTHS = range(1, 13)

# ==============================
# 2) دوال مساعدة
# ==============================

def generate_temperature(month: int) -> float:
    mean, std = MONTH_TEMP[month]
    temp = np.random.normal(mean, std)
    return round(float(np.clip(temp, 8, 50)), 1)

def generate_production_target(department: str, facility: str, year: int, month: int) -> float:
    """
    نضبط Production_Target حسب القسم + المنشأة + السنة + الموسم.
    """
    # base حسب القسم
    if "Well" in department:
        base = 180_000
    elif "Gas" in department:
        base = 95_000
    else:  # Maint
        base = 40_000

    # slight facility factor
    facility_factor = 1.0
    if "Ghawar" in facility:
        facility_factor = 1.15
    elif "Abqaiq" in facility:
        facility_factor = 1.05

    # ترند سنوي (كل سنة +2%)
    year_factor = 1 + 0.02 * (year - YEARS[0])

    # موسمية خفيفة
    seasonal = 1 + 0.05 * np.sin((month - 1) / 12 * 2 * np.pi)

    target = base * facility_factor * year_factor * seasonal
    return target

def generate_actual_from_target(target: float) -> float:
    """
    Actual قريب من Target بهامش ±5% تقريباً.
    """
    actual = np.random.normal(loc=target, scale=target * 0.04)
    return max(actual, 0)

def generate_energy_from_production(prod_target: float, prod_actual: float):
    """
    نفترض كثافة طاقة تقريبية (kWh لكل وحدة إنتاج).
    """
    intensity = 0.12  # تقديري
    energy_target = prod_target * intensity
    energy_actual = prod_actual * intensity * np.random.normal(1.0, 0.02)
    return energy_target, energy_actual

def generate_downtime_minutes(department: str) -> int:
    """
    Downtime شهري واقعي تقريباً حسب القسم.
    """
    if "Well" in department:
        base = np.random.exponential(scale=120)
    elif "Gas" in department:
        base = np.random.exponential(scale=180)
    else:  # Maint
        base = np.random.exponential(scale=300)
    return int(min(base, 3000))

def last_day_of_month(year: int, month: int) -> str:
    return (pd.Timestamp(year=year, month=month, day=1) + MonthEnd(1)).strftime("%Y-%m-%d")

def generate_equipment_id() -> str:
    return f"EQT-{random.randint(1000, 9999)}"

def pick_stream() -> str:
    return random.choice(STREAMS)

def pick_operator() -> str:
    return random.choice(OPERATOR_NAMES)

def pick_note() -> str:
    return random.choice(NOTES_LIST)


# ==============================
# 3) توليد الصفوف
# ==============================

rows = []

for year in YEARS:
    for month in MONTHS:
        for dept in DEPARTMENTS:
            for facility in FACILITIES:
                date_str = last_day_of_month(year, month)
                quarter = (month - 1) // 3 + 1

                prod_target = generate_production_target(dept, facility, year, month)
                prod_actual = generate_actual_from_target(prod_target)

                energy_target, energy_actual = generate_energy_from_production(
                    prod_target, prod_actual
                )

                temp_c = generate_temperature(month)
                downtime_min = generate_downtime_minutes(dept)

                variance = prod_actual - prod_target
                variance_pct = (variance / prod_target * 100) if prod_target > 0 else 0

                # نقدر نربط باقي القيَم كنِسَب من الإنتاج أو نخليها None لو ما تستخدمها
                compression_target = prod_target * 0.10
                compression_actual = compression_target * np.random.normal(1.0, 0.05)

                processing_target = prod_target * 0.30
                processing_actual = processing_target * np.random.normal(1.0, 0.05)

                refinery_target = prod_target * 0.40
                refinery_actual = refinery_target * np.random.normal(1.0, 0.05)

                throughput_target = prod_target * 0.85
                throughput_actual = throughput_target * np.random.normal(1.0, 0.03)

                yield_target = 0.94  # 94% مثلاً
                yield_actual = np.clip(
                    np.random.normal(yield_target, 0.02), 0.85, 0.99
                )

                rows.append({
                    "AdminArea": ADMIN_AREA,

                    "Compression_Actual": round(compression_actual, 2),
                    "Compression_Target": round(compression_target, 2),

                    "Date": date_str,
                    "Department": dept,
                    "Downtime_Min": downtime_min,

                    "Energy_Actual": round(energy_actual, 2),
                    "Energy_Target": round(energy_target, 2),

                    "Equipment_ID": generate_equipment_id(),
                    "Facility_Name": facility,

                    "Notes": pick_note(),
                    "Operator_Name": pick_operator(),

                    "Processing_Actual": round(processing_actual, 2),
                    "Processing_Target": round(processing_target, 2),

                    "Production_Actual": round(prod_actual, 3),
                    "Production_Target": round(prod_target, 0),

                    "Quarter": quarter,

                    "Refinery_Actual": round(refinery_actual, 2),
                    "Refinery_Target": round(refinery_target, 2),

                    "Stream": pick_stream(),
                    "Temperature_C": temp_c,

                    "Throughput_Actual": round(throughput_actual, 2),
                    "Throughput_Target": round(throughput_target, 2),

                    "Yield_Actual": round(yield_actual, 4),
                    "Yield_Target": round(yield_target, 4),
                })

df = pd.DataFrame(rows)

print(df.head())
print("Total rows:", len(df))

# ==============================
# 4) حفظ في SQLite
# ==============================

DB_PATH = "energyw.db"
TABLE_NAME = "upstream_metrics"

conn = sqlite3.connect(DB_PATH)
df.to_sql(TABLE_NAME, conn, if_exists="replace", index=False)
conn.close()

print(f"Saved {len(df)} records to {DB_PATH} table {TABLE_NAME}.")


                    AdminArea  Compression_Actual  Compression_Target  \
0  Upstream Admin Area - East            22276.34             20700.0   
1  Upstream Admin Area - East            19798.10             18900.0   
2  Upstream Admin Area - East            16717.73             18000.0   
3  Upstream Admin Area - East            17854.42             18000.0   
4  Upstream Admin Area - East            18154.23             18000.0   

         Date   Department  Downtime_Min  Energy_Actual  Energy_Target  \
0  2022-01-31  UP Well Ops            20       25263.48        24840.0   
1  2022-01-31  UP Well Ops            41       21752.06        22680.0   
2  2022-01-31  UP Well Ops           107       22763.07        21600.0   
3  2022-01-31  UP Well Ops            82       21573.70        21600.0   
4  2022-01-31  UP Well Ops            11       20532.97        21600.0   

  Equipment_ID         Facility_Name  ... Production_Target Quarter  \
0     EQT-2824           Ghawar Area  ...    

In [3]:

df.to_csv("data.csv", index=False)


In [5]:
import pandas as pd
import sqlite3

DB_PATH = "data1.db"
CSV_PATH = "data.csv"

df = pd.read_csv(CSV_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df["Quarter"] = df["Date"].dt.quarter

conn = sqlite3.connect(DB_PATH)

df.to_sql("upstream_metrics", conn, if_exists="append", index=False)
conn.close()

print("✔ Data inserted successfully into DB")


✔ Data inserted successfully into DB


In [20]:
from faker import Faker
import pandas as pd
import numpy as np
import sqlite3
import random, string

fake = Faker()

DB_PATH = "energyw.db"

# -------------------------------------------------------------
# 1) ADMIN AREAS & SHORT DEPARTMENTS (اختصار أسماء الأقسام)
# -------------------------------------------------------------
ADMIN_AREAS = {
    "Upstream Admin Area": [
        "UP Well Ops",
        "UP Field Prod",
        "UP Gas Inj",
        "UP Resv Mgmt",
        "UP Drilling",
        "UP Surf Fac"
    ],
    "Midstream Admin Area": [
        "MS Pipeline Ops",
        "MS Compression",
        "MS Processing",
        "MS Storage & Term",
        "MS Metering",
        "MS Integrity"
    ],
    "Downstream Admin Area": [
        "DS Refining",
        "DS Petrochem",
        "DS Distribution",
        "DS Export Term",
        "DS Blending",
        "DS Utilities"
    ]
}

# -------------------------------------------------------------
# 2) DYNAMIC STREAM GENERATOR
# -------------------------------------------------------------
def gen_streams(area):
    prefix = {
        "Upstream Admin Area": "UP",
        "Midstream Admin Area": "MS",
        "Downstream Admin Area": "DS"
    }[area]

    stream_count = random.randint(3, 6)

    streams = set()
    while len(streams) < stream_count:
        code = f"{prefix}-{random.choice(string.ascii_uppercase)}{random.randint(10,99)}"
        streams.add(code)

    return list(streams)

# -------------------------------------------------------------
# 3) REALISTIC VARIANCE
# -------------------------------------------------------------
def apply_variance(target, low=-0.25, high=0.35):
    if target is None:
        return None
    return round(target * (1 + np.random.uniform(low, high)), 3)

# -------------------------------------------------------------
# 4) OIL MARKET CONTEXT (سعر النفط حسب السنة والشهر)
# -------------------------------------------------------------
YEAR_BASE_OIL = {
    2020: 43.0,
    2021: 71.0,
    2022: 99.0,
    2023: 82.0,
    2024: 80.0,
    2025: 62.0,
}

def get_oil_price_for_month(date: pd.Timestamp) -> float:
    year = int(date.year)
    month = int(date.month)
    base_price = YEAR_BASE_OIL.get(year, 70.0)

    factor = 1.0

    if year == 2020:
        if month in (4, 5):
            factor -= 0.35
        elif month in (3, 6):
            factor -= 0.20

    if year == 2022 and month in (3, 4, 5, 6):
        factor += 0.15

    factor += np.random.normal(0, 0.03)

    price = max(20.0, base_price * factor)
    return round(price, 2)

# -------------------------------------------------------------
# 5) LINEAR ENERGY MODEL
# -------------------------------------------------------------
def compute_energy_target(area,
                          prod_target,
                          throughput_target,
                          compression_target,
                          processing_target,
                          refinery_target,
                          yield_target,
                          temp_c,
                          oil_price):
    base = 0.0

    if "Upstream" in area and prod_target is not None:
        base = prod_target * 0.12

    elif "Midstream" in area:
        base = (
            0.02 * (throughput_target or 0) +
            150 * (compression_target or 0) +
            0.015 * (processing_target or 0)
        )

    elif "Downstream" in area:
        base = (
            0.06 * (refinery_target or 0) +
            400 * (yield_target or 0)
        )

    temp_factor = 1.0 + 0.01 * ((temp_c - 100.0) / 20.0)

    if oil_price < 50:
        market_factor = 0.95
    elif oil_price > 90:
        market_factor = 1.05
    else:
        market_factor = 1.00

    energy_target = base * temp_factor * market_factor

    energy_target = max(5_000, min(90_000, energy_target))
    return round(energy_target, 2)

def compute_energy_actual(energy_target):
    if energy_target is None:
        return None
    noise = np.random.normal(1.0, 0.03)
    val = energy_target * noise
    return int(round(val))

# -------------------------------------------------------------
# 6) BUILD STREAM CATALOG
# -------------------------------------------------------------
def build_stream_catalog():
    catalog = []
    for area, departments in ADMIN_AREAS.items():
        for dept in departments:
            streams = gen_streams(area)
            for stream in streams:
                catalog.append({
                    "AdminArea": area,
                    "Department": dept,
                    "Stream": stream,
                    "Operator_Name": fake.name(),
                    "Facility_Name": fake.company(),
                    "Equipment_ID": fake.bothify("EQT-####"),
                    "Notes": fake.sentence(nb_words=10),
                })
    return catalog

# -------------------------------------------------------------
# 7) EQUIPMENT TEMPERATURE MODEL (احترافي)
# -------------------------------------------------------------
def compute_equipment_temperature(area,
                                  stream,
                                  date,
                                  prod_target,
                                  throughput_target,
                                  compression_target,
                                  processing_target,
                                  refinery_target,
                                  yield_target,
                                  prev_temp):

    if "Upstream" in area:
        base = 80.0
        min_t, max_t = 50.0, 120.0
    elif "Midstream" in area:
        base = 70.0
        min_t, max_t = 40.0, 110.0
    else:
        base = 150.0
        min_t, max_t = 90.0, 250.0

    stream_bias = (hash(stream) % 11 - 5) * 1.2

    load_index = 0.0
    if prod_target:
        load_index += (prod_target - 150000) / 200000.0
    if throughput_target:
        load_index += (throughput_target - 600000) / 400000.0
    if processing_target:
        load_index += (processing_target - 200000) / 300000.0
    if refinery_target:
        load_index += (refinery_target - 320000) / 200000.0
    if yield_target:
        load_index += (yield_target - 70.0) / 25.0
    if compression_target:
        load_index += (compression_target - 80.0) / 20.0

    load_index = max(-1.5, min(2.0, load_index))

    load_effect = 10.0 * load_index
    noise = np.random.normal(0, 3.0)

    target_temp = base + stream_bias + load_effect + noise

    if prev_temp is None:
        temp = target_temp
    else:
        temp = 0.7 * prev_temp + 0.3 * target_temp

    temp = max(min_t, min(max_t, temp))
    return round(temp, 2)

# -------------------------------------------------------------
# 8) MAIN DATA GENERATOR — 2020 → 2025
# -------------------------------------------------------------
def generate_data():
    rows = []

    dates = pd.date_range("2020-01-01", "2025-09-30", freq="M")
    catalog = build_stream_catalog()
    prev_temp_by_stream = {}

    record_id = 1  # <<< تمت إضافة ID هنا

    for date in dates:
        month = date.month
        quarter = (month - 1) // 3 + 1
        oil_price = get_oil_price_for_month(date)

        for meta in catalog:
            area = meta["AdminArea"]
            dept = meta["Department"]
            stream = meta["Stream"]

            prod_target = prod_actual = None
            throughput_target = throughput_actual = None
            compression_target = compression_actual = None
            processing_target = processing_actual = None
            refinery_target = refinery_actual = None
            yield_target = yield_actual = None

            if "Upstream" in area:
                prod_target = random.randint(150000, 350000)
                prod_actual = apply_variance(prod_target, -0.18, 0.22)

            elif "Midstream" in area:
                throughput_target = random.randint(600000, 950000)
                throughput_actual = apply_variance(throughput_target, -0.15, 0.18)

                compression_target = round(random.uniform(82, 96), 2)
                compression_actual = apply_variance(compression_target, -0.06, 0.06)

                processing_target = random.randint(200000, 500000)
                processing_actual = apply_variance(processing_target, -0.15, 0.18)

            else:
                refinery_target = random.randint(320000, 520000)
                refinery_actual = apply_variance(refinery_target, -0.12, 0.18)

                yield_target = round(random.uniform(70, 96), 2)
                yield_actual = apply_variance(yield_target, -0.05, 0.05)

            prev_temp = prev_temp_by_stream.get(stream)
            temp_c = compute_equipment_temperature(
                area,
                stream,
                date,
                prod_target,
                throughput_target,
                compression_target,
                processing_target,
                refinery_target,
                yield_target,
                prev_temp
            )
            prev_temp_by_stream[stream] = temp_c

            energy_target = compute_energy_target(
                area,
                prod_target,
                throughput_target,
                compression_target,
                processing_target,
                refinery_target,
                yield_target,
                temp_c,
                oil_price
            )
            energy_actual = compute_energy_actual(energy_target)

            rows.append({
                "ID": record_id,   
                "Date": date,
                "AdminArea": area,
                "Department": dept,
                "Stream": stream,

                "Operator_Name": meta["Operator_Name"],
                "Facility_Name": meta["Facility_Name"],
                "Equipment_ID": meta["Equipment_ID"],
                "Notes": meta["Notes"],

                "Production_Target": prod_target,
                "Production_Actual": prod_actual,
                "Throughput_Target": throughput_target,
                "Throughput_Actual": throughput_actual,
                "Compression_Target": compression_target,
                "Compression_Actual": compression_actual,
                "Processing_Target": processing_target,
                "Processing_Actual": processing_actual,
                "Refinery_Target": refinery_target,
                "Refinery_Actual": refinery_actual,
                "Yield_Target": yield_target,
                "Yield_Actual": yield_actual,

                "Energy_Target": energy_target,
                "Energy_Actual": energy_actual,

                "Temperature_C": temp_c,
                "Downtime_Min": random.choice([0, 3, 5, 10, 20, 60, 90, 120]),
                "Quarter": quarter
            })

            record_id += 1

    return pd.DataFrame(rows)

# -------------------------------------------------------------
# 9) SAVE DATA
# -------------------------------------------------------------
def save(df):
    conn = sqlite3.connect(DB_PATH)
    df.to_sql("upstream_metrics", conn, if_exists="replace", index=False)
    conn.close()
    print("Saved", len(df), "rows.")

# -------------------------------------------------------------
# 10) RUN
# -------------------------------------------------------------
if __name__ == "__main__":
    df = generate_data()
    print(df.head(5).to_dict(orient="records"))
    save(df)


C:\Users\ibrahem\AppData\Local\Temp\ipykernel_2116\2161986217.py:233: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range("2020-01-01", "2025-09-30", freq="M")


[{'ID': 1, 'Date': Timestamp('2020-01-31 00:00:00'), 'AdminArea': 'Upstream Admin Area', 'Department': 'UP Well Ops', 'Stream': 'UP-A22', 'Operator_Name': 'Michelle Wilson', 'Facility_Name': 'Walker LLC', 'Equipment_ID': 'EQT-4722', 'Notes': 'Lead season shake its agency race light remember black whatever together society.', 'Production_Target': 256729.0, 'Production_Actual': 228609.579, 'Throughput_Target': nan, 'Throughput_Actual': nan, 'Compression_Target': nan, 'Compression_Actual': nan, 'Processing_Target': nan, 'Processing_Actual': nan, 'Refinery_Target': nan, 'Refinery_Actual': nan, 'Yield_Target': nan, 'Yield_Actual': nan, 'Energy_Target': 29083.89, 'Energy_Actual': 28783, 'Temperature_C': 87.48, 'Downtime_Min': 5, 'Quarter': 1}, {'ID': 2, 'Date': Timestamp('2020-01-31 00:00:00'), 'AdminArea': 'Upstream Admin Area', 'Department': 'UP Well Ops', 'Stream': 'UP-U63', 'Operator_Name': 'John Robles', 'Facility_Name': 'Greene, Rubio and Hernandez', 'Equipment_ID': 'EQT-3428', 'Notes'

In [4]:
df

,Date,AdminArea,Department,Stream,Operator_Name,Facility_Name,Equipment_ID,Notes,Production_Target,Production_Actual,...,Processing_Actual,Refinery_Target,Refinery_Actual,Yield_Target,Yield_Actual,Energy_Target,Energy_Actual,Temperature_C,Downtime_Min,Quarter
0,2020-01-31,Upstream Admin Area,UP Well Ops,UP-Y93,Angela Fisher,Dean-Johnson,EQT-0582,Significant affect visit sell power decision i...,263119.0,222730.765,...,NaN,NaN,NaN,NaN,NaN,29702.21,30148,80.44,10,1
1,2020-01-31,Upstream Admin Area,UP Well Ops,UP-U15,Melissa Everett,Logan PLC,EQT-7941,Sing least dinner include record east your wid...,282563.0,331740.563,...,NaN,NaN,NaN,NaN,NaN,32021.16,31991,88.14,10,1
2,2020-01-31,Upstream Admin Area,UP Well Ops,UP-N23,Denise Reed,"Thomas, Guerrero and Foster",EQT-4113,Reason choice personal teacher including trave...,310037.0,265803.864,...,NaN,NaN,NaN,NaN,NaN,35208.85,35679,92.34,120,1
3,2020-01-31,Upstream Admin Area,UP Well Ops,UP-W53,Krystal Coffey,King and Sons,EQT-0044,Book push reduce interview trouble any allow b...,181445.0,210243.534,...,NaN,NaN,NaN,NaN,NaN,20424.31,20787,74.82,0,1
4,2020-01-31,Upstream Admin Area,UP Well Ops,UP-R48,Dr. Gina Collins,"Moore, Potts and Patrick",EQT-3363,Behavior provide walk practice physical preven...,224428.0,241576.745,...,NaN,NaN,NaN,NaN,NaN,25438.06,25247,88.53,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5998,2025-09-30,Downstream Admin Area,DS Utilities,DS-R69,Timothy Gonzalez,"Martinez, Aguilar and Spence",EQT-6383,Audience after well product green say shake of.,NaN,NaN,...,NaN,416377.0,395766.409,72.20,68.948,55574.64,57267,163.57,3,3
5999,2025-09-30,Downstream Admin Area,DS Utilities,DS-M71,Matthew Wang,Oneill LLC,EQT-8174,Home theory wrong huge way close baby stuff on...,NaN,NaN,...,NaN,330459.0,345954.138,72.70,75.364,50111.15,50865,149.22,120,3
6000,2025-09-30,Downstream Admin Area,DS Utilities,DS-O95,Ryan White DVM,"Bailey, Heath and Mitchell",EQT-0963,Store stay Congress back early wide until shar...,NaN,NaN,...,NaN,334172.0,329936.632,72.03,70.270,50370.21,53012,161.72,60,3
6001,2025-09-30,Downstream Admin Area,DS Utilities,DS-F31,Donald Kennedy,"Velazquez, Stanton and Hammond",EQT-2392,South camera subject source spring fear because.,NaN,NaN,...,NaN,519744.0,512752.627,75.34,76.217,63135.73,64587,159.20,3,3
